In [24]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


d:\University of Calgary\Winter 2026\ENGG681-Engineering-Tools\ENGG681-Assignment-Codes\venv\Scripts\python.exe: No module named pip


# Problem 2

Here, we will use python and numpy to solve the problem. Numpy just provides more precision and ease of matrix-vector operations. Similar libraries are available in almost all modern programming languages

In [25]:
import numpy as np

The given data for s and t are as follows:

In [26]:
s_values = np.array([32, 54, 61, 75, 82, 93, 105, 118, 131, 157], dtype=float)
t_values = np.array([854, 912, 1003, 1012, 1125, 955, 1005, 896, 912, 802], dtype=float)

query_s_values = np.array([40, 65, 110, 140], dtype=float)

In [27]:
def get_interval_index(
    s_values: np.ndarray, query_s_value: float
) -> np.signedinteger | int:
    """
    A utility function to find the appropriate interval index for a given query s value.
     - If query_s_value is less than or equal to the first s value, return 0 (use the first interval).
     - If query_s_value is greater than or equal to the last s value, return len(s_values) - 2 (use the last interval).
     - Otherwise, find the correct interval using np.searchsorted and return the index.
    """

    if query_s_value <= s_values[0]:
        return 0
    elif query_s_value >= s_values[-1]:
        return len(s_values) - 2
    else:
        i = np.searchsorted(s_values, query_s_value, side="right") - 1
        return min(i, len(s_values) - 2)

## a) Lagrange Interpolation 

Lagrange interpolation uses all 10 given data points to build a single degree-9 polynomial. Each value is computed as a weighted sum of all known t values.

In [28]:
def lagrange_interpolate(
    query_s_value: float, s_values: np.ndarray, t_values: np.ndarray
) -> float:
    n = len(s_values)
    result = 0.0
    for i in range(n):
        # Compute the i-th Lagrange basis polynomial L_i(s_val)
        L_i = 1.0
        for j in range(n):
            if j != i:
                L_i *= (query_s_value - s_values[j]) / (s_values[i] - s_values[j])
        result += t_values[i] * L_i
    return result

The interpolating points s = 40 and s = 140 are near the boundary of the given data range. Since the polynomial constructed from the data points have a very high degree of 9, it oscillates near the boundaries of the data points, hence we can see a very high jump in interpolated t values (-4607.523554 and 7695.620984 respectively) at these points. 

In [29]:
print("(a) Lagrange Interpolation Results:")
print("=" * 55)
print(f"{'query s value':>10}  {'t (interpolated)':>20}")
print("-" * 35)

for query in query_s_values:
    interpolated_t_value = lagrange_interpolate(
        query_s_value=query,
        s_values=s_values,
        t_values=t_values,
    )
    print(f"{query:>10}  {interpolated_t_value:>20.6f}")

(a) Lagrange Interpolation Results:
query s value      t (interpolated)
-----------------------------------
      40.0          -4607.523554
      65.0            910.730245
     110.0           1106.869258
     140.0           7695.620984


## b) Quadratic Spline Interpolation

For this problem with 10 data points (9 intervals), and quadratic spline, each interval i has a quadratic: a_i + b_i*(s - s_i) + c_i*(s - s_i)^2. So, total unknowns would be 9*3 = 27. Since each interval has it's own polynomial, the interpolation is much smoother here and is not subjected to Runge conditions like in Lagrange interpolation.

Now, for the 4 conditions: 

1. Each spline passes through left knot:  a_i = t_data[i] -> 9 equations
2. Each spline passes through right knot: spline_i(s_{i+1}) -> t_{i+1} -> 9 equations
3. Interior derivatives match (continuity of 1st derivative) -> (9-1) = 8 equations
4. Natural condition: c_0 = 0 (first spline is linear) -> 1 equation

In [30]:
def get_quadratic_spline_coeffs(
    s_values: np.ndarray, t_values: np.ndarray
) -> np.ndarray:
    """This function computes the coefficients of a piecewise quadratic spline that interpolates the given data points (s_values, t_values). This essentially sets up and solves a linear system based on the conditions for quadratic splines."""

    num_intervals = len(s_values) - 1  # number of intervals = total points - 1

    num_unknowns = 3 * num_intervals
    A = np.zeros((num_unknowns, num_unknowns))
    b_vec = np.zeros(num_unknowns)

    # Index layout: for interval i → [a_i, b_i, c_i] at columns [3i, 3i+1, 3i+2]

    row = 0

    # Condition 1: a_i = t_values[i]
    for i in range(num_intervals):
        A[row, 3 * i] = 1.0
        b_vec[row] = t_values[i]
        row += 1

    # Condition 2: a_i + b_i*h_i + c_i*h_i^2 = t_values[i+1]
    for i in range(num_intervals):
        h = s_values[i + 1] - s_values[i]
        A[row, 3 * i] = 1.0
        A[row, 3 * i + 1] = h
        A[row, 3 * i + 2] = h**2
        b_vec[row] = t_values[i + 1]
        row += 1

    # Condition 3: Interior first-derivative continuity
    # b_i + 2*c_i*h_i = b_{i+1}  →  b_i + 2*c_i*h_i - b_{i+1} = 0
    for i in range(num_intervals - 1):
        h = s_values[i + 1] - s_values[i]
        A[row, 3 * i + 1] = 1.0
        A[row, 3 * i + 2] = 2.0 * h
        A[row, 3 * (i + 1) + 1] = -1.0
        b_vec[row] = 0.0
        row += 1

    # Condition 4: c_0 = 0
    A[row, 2] = 1.0
    b_vec[row] = 0.0
    row += 1

    # Solve the system
    coeffs = np.linalg.solve(A, b_vec)

    return coeffs


def evaluate_quadratic_spline(
    s_data: np.ndarray, coeffs: np.ndarray, s_val: float
) -> float:

    i = get_interval_index(s_data, s_val)

    a_i = coeffs[3 * i]
    b_i = coeffs[3 * i + 1]
    c_i = coeffs[3 * i + 2]
    ds = s_val - s_data[i]

    return a_i + b_i * ds + c_i * ds**2

In [31]:
coeffs = get_quadratic_spline_coeffs(s_values, t_values)

print("Quadratic Spline Interpolation")
print("=" * 55)
print(f"{'s':>10}  {'t (interpolated)':>20}")
print("-" * 35)
for query in query_s_values:
    interpolated_t_value = evaluate_quadratic_spline(s_values, coeffs, query)
    print(f"{query:>10}  {interpolated_t_value:>20.6f}")

Quadratic Spline Interpolation
         s      t (interpolated)
-----------------------------------
      40.0            875.090909
      65.0           1070.487941
     110.0           1276.894388
     140.0           1562.820871
